In [1]:
from google.colab import drive
drive.mount('/content/drive')
!df -h /content | tail -1   # cần còn trống >= 45 GB

Mounted at /content/drive
overlay         226G   21G  206G  10% /


In [2]:
import os, json, shutil, re, csv, tarfile
from pathlib import Path
from google.colab import files

USER = "hoangleev"
SLUG = f"{USER}/fatformer-test-datasets"
TITLE = "FatFormer Test Datasets"

DATA_DIR  = Path("/content/drive/MyDrive/Fatformer/datasets")  # sửa nếu đường dẫn khác
CLEAN_DIR = Path("/content/clean")
STAGE     = Path("/content/kaggle_upload")
LOG_DIR   = Path("/content/rename_logs")
for d in (CLEAN_DIR, LOG_DIR): d.mkdir(exist_ok=True)

for n in ["test_benchmark_diffusion.tar", "test_benchmark_gans.tar", "test_degraded.tar"]:
    assert (DATA_DIR / n).is_file(), f"Không thấy {n} trong {DATA_DIR}"

!pip install -q --upgrade kaggle
up = files.upload()   # chọn kaggle.json
creds = json.loads(next(iter(up.values())))
assert creds["username"] == USER, f"kaggle.json thuộc {creds['username']}, không phải {USER}"
os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"] = creds["key"]
kd = Path.home() / ".kaggle"; kd.mkdir(exist_ok=True)
(kd / "kaggle.json").write_text(json.dumps(creds)); os.chmod(kd / "kaggle.json", 0o600)
print("OK, đã đăng nhập Kaggle:", USER)

Saving kaggle.json to kaggle.json
OK, đã đăng nhập Kaggle: hoangleev


In [3]:
BAD = re.compile(r'[:*?"<>|\\]')

def clean_tar(name):
    src, dst = DATA_DIR / name, CLEAN_DIR / name
    if dst.exists(): dst.unlink()
    seen, renamed, count = set(), [], 0
    with tarfile.open(src, "r|") as tin, \
         tarfile.open(dst, "w|", format=tarfile.PAX_FORMAT) as tout:
        for m in tin:
            old = m.name
            new = "/".join(BAD.sub("-", p) for p in old.split("/"))
            if m.isfile():
                k = 1
                while new in seen:                      # tránh trùng tên sau khi đổi
                    stem, ext = os.path.splitext(new)
                    new = f"{stem}_dup{k}{ext}"; k += 1
                seen.add(new)
            if new != old:
                renamed.append((old, new))
            m.name = new
            tout.addfile(m, tin.extractfile(m) if m.isfile() else None)
            count += 1
            if count % 50000 == 0:
                print(f"  ...{count} mục", flush=True)
    with open(LOG_DIR / f"{name}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["old", "new"]); w.writerows(renamed)
    print(f"[{name}] tổng {count} mục, đã đổi tên {len(renamed)} (log: rename_logs/{name}.csv)")
    print("  ví dụ:", renamed[:2])

def prepare(names):
    if STAGE.exists(): shutil.rmtree(STAGE)
    STAGE.mkdir(parents=True)
    for n in names:
        try: os.link(CLEAN_DIR / n, STAGE / n)          # hard link, không tốn thêm đĩa
        except OSError: shutil.copy(CLEAN_DIR / n, STAGE / n)
    meta = {"title": TITLE, "id": SLUG, "licenses": [{"name": "CC0-1.0"}]}
    (STAGE / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    print("Staging:", sorted(p.name for p in STAGE.iterdir()))

In [4]:
clean_tar("test_benchmark_diffusion.tar")
prepare(["test_benchmark_diffusion.tar"])
!kaggle datasets create -p /content/kaggle_upload

[test_benchmark_diffusion.tar] tổng 20033 mục, đã đổi tên 0 (log: rename_logs/test_benchmark_diffusion.tar.csv)
  ví dụ: []
Staging: ['dataset-metadata.json', 'test_benchmark_diffusion.tar']
Starting upload for file test_benchmark_diffusion.tar
100% 1.22G/1.22G [00:57<00:00, 22.6MB/s]
Upload successful: test_benchmark_diffusion.tar (1GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/hoangleev/fatformer-test-datasets


In [5]:
clean_tar("test_benchmark_gans.tar")
prepare(["test_benchmark_diffusion.tar", "test_benchmark_gans.tar"])
!kaggle datasets version -p /content/kaggle_upload -m "Add gans test set"

  ...50000 mục
[test_benchmark_gans.tar] tổng 90460 mục, đã đổi tên 1094 (log: rename_logs/test_benchmark_gans.tar.csv)
  ví dụ: [('test/cyclegan/summer/1_fake/2015-12-31 17:23:30_fake.png', 'test/cyclegan/summer/1_fake/2015-12-31 17-23-30_fake.png'), ('test/cyclegan/summer/1_fake/2011-12-29 09:02:31_fake.png', 'test/cyclegan/summer/1_fake/2011-12-29 09-02-31_fake.png')]
Staging: ['dataset-metadata.json', 'test_benchmark_diffusion.tar', 'test_benchmark_gans.tar']
Starting upload for file test_benchmark_diffusion.tar
100% 1.22G/1.22G [00:59<00:00, 22.0MB/s]
Upload successful: test_benchmark_diffusion.tar (1GB)
Starting upload for file test_benchmark_gans.tar
100% 18.7G/18.7G [13:45<00:00, 24.4MB/s]
Upload successful: test_benchmark_gans.tar (19GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/hoangleev/fatformer-test-datasets


In [6]:
clean_tar("test_degraded.tar")
prepare(["test_benchmark_diffusion.tar", "test_benchmark_gans.tar", "test_degraded.tar"])
!kaggle datasets version -p /content/kaggle_upload -m "Add degraded test set"

  ...50000 mục
  ...100000 mục
  ...150000 mục
  ...200000 mục
  ...250000 mục
  ...300000 mục
  ...350000 mục
  ...400000 mục
  ...450000 mục
  ...500000 mục
  ...550000 mục
  ...600000 mục
  ...650000 mục
[test_degraded.tar] tổng 662953 mục, đã đổi tên 6564 (log: rename_logs/test_degraded.tar.csv)
  ví dụ: [('test_degraded/jpeg_q70/cyclegan/winter/0_real/2015-01-03 18:27:31_real.jpg', 'test_degraded/jpeg_q70/cyclegan/winter/0_real/2015-01-03 18-27-31_real.jpg'), ('test_degraded/jpeg_q70/cyclegan/winter/0_real/2012-12-25 14:28:50_real.jpg', 'test_degraded/jpeg_q70/cyclegan/winter/0_real/2012-12-25 14-28-50_real.jpg')]
Staging: ['dataset-metadata.json', 'test_benchmark_diffusion.tar', 'test_benchmark_gans.tar', 'test_degraded.tar']
Starting upload for file test_benchmark_diffusion.tar
100% 1.22G/1.22G [00:54<00:00, 24.0MB/s]
Upload successful: test_benchmark_diffusion.tar (1GB)
Starting upload for file test_degraded.tar
100% 19.5G/19.5G [14:28<00:00, 24.2MB/s]
Upload successful: test_d

In [7]:
!kaggle datasets status hoangleev/fatformer-eval-benchmarks
!kaggle datasets files hoangleev/fatformer-eval-benchmarks --page-size 10

readyNext Page Token = CfDJ8FwkkMkekkBCtzoFDl4Vj7gV7Ejrj_wCYn6DaFDZ4yAUoiw93Qqk4o6gwzqvdUcAUKUfrLRuklxRma4Q_Qo7KefwBQykXtVbDNN35EULagEFa9vmxU1-t8qfxvlBlSqZnqFZAIevBKkS23WtbASaQ5iMPXwBYQjg4vEi2BakBBCDdGeW
name                                         size  creationDate                
------------------------------------------  -----  --------------------------  
diffusion_test/dalle/0_real/aadlygmazf.jpg  14738  2026-10-25 12:01:12.962000  
diffusion_test/dalle/0_real/aahxgrlfqt.jpg  12626  2026-10-25 12:01:14.651000  
diffusion_test/dalle/0_real/aaksjqrtil.jpg  10876  2026-10-25 12:01:14.433000  
diffusion_test/dalle/0_real/aakydlwfer.jpg  21438  2026-10-25 12:01:14.026000  
diffusion_test/dalle/0_real/abaxzegfoi.jpg  28726  2026-10-25 12:01:13.076000  
diffusion_test/dalle/0_real/abzfomfxjd.jpg  46621  2026-10-25 12:01:14.458000  
diffusion_test/dalle/0_real/acpwmtyztu.jpg  27933  2026-10-25 12:01:14.733000  
diffusion_test/dalle/0_real/adizyjvxam.jpg  18314  2026-10-25 12:01:13.43900